# User-friendly toy generation

This notebook shows the short pseudo-data API once a `DecayModel` already exists. The helpers generate the phase-space proposal pool, compute the dynamical weights, apply efficiency/vetoes, resample unweighted events and optionally add backgrounds.

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    CPToyBackground, CPRealImag, DecayChannel, DecayModel, NonResonant,
    Parameter, RealImag, Resonance, ToyBackground, enable_x64,
    generate_cp_toy, generate_toy, plot_dalitz, plot_square_dalitz,
)

enable_x64()

## 1. Build a normal B+ model

In [ ]:
channel = DecayChannel('B+', ('K+', 'pi+', 'pi-'))
model = DecayModel(
    channel,
    [
        Resonance('Kstar892', (0,2), RealImag(1.0, 0.0), mass=0.8958, width=0.0474, spin=1),
        Resonance('rho770', (1,2), RealImag(0.55, 0.35), mass=0.7753, width=0.1491, spin=1),
        NonResonant(RealImag(0.35, -0.20)),
    ],
    normalization_method='square-dalitz',
    normalization_resolution=120,
    normalization_pair=(0,2),
)

## 2. Signal + efficiency + background in one call

The helper below replaces manual `generate_phase_space`, intensity evaluation, weighted resampling and event merging.

In [ ]:
efficiency = lambda d: 0.55 + 0.35*jnp.clip(d['s13']/25.0, 0.0, 1.0)
background = ToyBackground(
    'combinatorial',
    lambda d: 0.4 + 0.7*jnp.clip(d['s23']/20.0, 0.0, 1.0),
)

toy = generate_toy(
    model,
    20_000,
    efficiency=efficiency,
    signal_fraction=0.82,
    backgrounds=(background,),
    seed=1801,
    pool_size=120_000,
)
print('generated events:', toy.size)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
plot_dalitz(toy, x='s13', y='s23', bins=70, ax=axes[0], title='Generated Dalitz')
plot_square_dalitz(
    toy,
    mother_mass=channel.parent_mass,
    masses=channel.daughter_masses,
    pair=(0,2),
    bins=70,
    ax=axes[1],
    title='Generated Square Dalitz',
)

## 3. Direct-CP toy generation

`generate_cp_toy` computes the accepted B+/B- signal integrals and uses them to draw the charge split automatically. Background charge fractions are treated analogously.

In [ ]:
x = Parameter.coefficient('NR.x', 1.0, owner='NR')
dx = Parameter.coefficient('NR.dx', 0.12, owner='NR')
cp = CPRealImag(x, 0.0, dx, 0.0)

plus_model = DecayModel(
    DecayChannel('B+', ('K+', 'pi+', 'pi-')),
    [NonResonant(cp.for_charge(+1))],
    normalization_method='square-dalitz', normalization_resolution=100, normalization_pair=(0,2),
)
minus_model = DecayModel(
    DecayChannel('B-', ('K-', 'pi-', 'pi+')),
    [NonResonant(cp.for_charge(-1))],
    normalization_method='square-dalitz', normalization_resolution=100, normalization_pair=(0,2),
)

cp_background = CPToyBackground('comb', lambda d: jnp.ones_like(d['s12']))
plus_toy, minus_toy = generate_cp_toy(
    plus_model, minus_model, 20_000,
    parameters={'NR.x': 1.0, 'NR.dx': 0.12},
    signal_fraction=0.85,
    backgrounds=(cp_background,),
    seed=1802,
    pool_size=100_000,
)
print('B+ events:', plus_toy.size)
print('B- events:', minus_toy.size)
print('total:', plus_toy.size + minus_toy.size)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
plot_dalitz(plus_toy, x='s13', y='s23', bins=65, ax=axes[0], title='B+ toy')
plot_dalitz(minus_toy, x='s13', y='s23', bins=65, ax=axes[1], title='B- toy')

## 4. Minimal forms

Signal-only generation is simply:

```python
toy = generate_toy(model, 50_000, seed=1)
```

For a fitted parameter point, pass the mapping returned by the fit:

```python
toy = generate_toy(model, 50_000, parameters=fit_values, seed=2)
```